In [2]:
!pip install pymupdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 4.2 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip available: 22.3 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [3]:
import fitz  # PyMuPDF
import re

In [8]:
def extract_w2_fields(pdf_path):
    doc = fitz.open(pdf_path)
    page = doc[0]

    elements = []
    for block in page.get_text("dict")["blocks"]:
        for line in block.get("lines", []):
            for span in line.get("spans", []):
                elements.append({
                    "text": span["text"],
                    "x0": span["bbox"][0],
                    "y0": span["bbox"][1],
                    "x1": span["bbox"][2],
                    "y1": span["bbox"][3]
                })

    def find_value_near_label(label_keywords, max_y_gap=10, right_only=True):
        for el in elements:
            if any(re.search(kw, el["text"], re.IGNORECASE) for kw in label_keywords):
                label_y = el["y0"]
                label_x = el["x1"]
                candidates = []
                for e in elements:
                    y_diff = abs(e["y0"] - label_y)
                    x_diff = e["x0"] - label_x
                    if y_diff <= max_y_gap and (not right_only or x_diff > 0):
                        candidates.append((y_diff + x_diff, e))
                if candidates:
                    candidates.sort(key=lambda x: x[0])
                    return candidates[0][1]["text"]
        return None
    
    for e in elements:
        print(f"{e['text']:40} x={e['x0']:.1f} y={e['y0']:.1f}")

    result = {
        "employee_name": find_value_near_label([r"employee'?s? name"]),
        "ssn": find_value_near_label([r"social security number"]),
        "wages": find_value_near_label([r"wages, tips", r"wages"]),
        "federal_income_tax_withheld": find_value_near_label([r"federal income tax"]),
        "social_security_wages": find_value_near_label([r"social security wages"]),
        "medicare_wages": find_value_near_label([r"medicare wages"]),
        "employer_name": find_value_near_label([r"employer'?s? name"]),
        "employer_id": find_value_near_label([r"employer identification number", r"ein"])
    }

    doc.close()
    return result

# Example usage
pdf_file = "w2_sk.pdf"
data = extract_w2_fields(pdf_file)
print(data)


2024 W-2 and EARNINGS SUMMARY            x=279.0 y=6.8
Social Security Number:                  x=460.0 y=301.7
XXX-XX-3258                              x=535.0 y=301.7
PAGE 1 OF 1                              x=358.0 y=383.3
2024                                     x=238.5 y=378.7
d    Control  number                     x=9.0 y=67.7
Dept.                                    x=82.0 y=67.7
Corp.                                    x=112.0 y=67.7
Employer  use  only                      x=141.0 y=67.7
c  Employer's  name,  address,  and  ZIP  code x=9.0 y=86.7
e/f  Employee's  name,  address,  and  ZIP  code x=9.0 y=154.7
b   Employer's  FED  ID  number          x=9.0 y=202.7
a  Employee's  SSA  number               x=107.0 y=202.7
1  Wages, tips, other comp.              x=9.0 y=216.7
2  Federal income tax withheld           x=107.0 y=216.7
3  Social security wages                 x=9.0 y=233.7
4  Social security tax withheld          x=107.0 y=233.7
5  Medicare wages and tips           